# 🤖 True Agentic RAG with Microsoft AutoGen + Grok

### The Microsoft Philosophy: Conversation as State

Unlike LangGraph, which passes a single **State Dictionary** from node to node, the Microsoft AutoGen approach is fundamentally **conversational and event-driven**:

| Concept | LangGraph | Microsoft AutoGen |
|---------|-----------|-------------------|
| **State** | Dict passed between nodes | Chat history shared by all agents |
| **Routing** | Graph edges / conditional branches | Python FSM speaker-selection function |
| **Agents** | Function nodes | Distinct LLM personas that literally talk to each other |
| **Loop** | Cycle back to a node | Judge says `REJECT` → speaker-selector routes back to Retriever |

This notebook builds the exact same **Evaluator-Optimizer RAG Loop** you saw in `agentic-rag-demo.ipynb`, but in the AutoGen framework:

```
User Proxy ──► Retriever Agent (tool: search_database)
                    │
                    ▼
             Generator Agent (writes draft from chat history)
                    │
                    ▼
               Judge Agent ──── REJECT ──► Retriever (loop!)
                    │
                  APPROVE
                    │
                    ▼
                  END
```

**Requirements:** An [xAI API key](https://console.x.ai/) (free tier available). No local GPU needed.

---

In [ ]:
# CELL 1: Setup & Installations
# pyautogen brings the Microsoft AutoGen framework
# chromadb + sentence-transformers = local vector store (no API key needed for embeddings)
!pip install -qU pyautogen chromadb sentence-transformers langchain-huggingface langchain-community

In [ ]:
# CELL 2: Environment & LLM Configuration
import os
from getpass import getpass

# BYOK: Enter your xAI API Key
# Get one free at https://console.x.ai/
os.environ["XAI_API_KEY"] = getpass("Enter your xAI API Key: ")

# AutoGen natively speaks OpenAI's API shape.
# xAI is fully compatible — we just swap in the base_url.
grok_config = {
    "config_list": [{
        "model": "grok-2-latest",
        "api_key": os.environ["XAI_API_KEY"],
        "base_url": "https://api.x.ai/v1"
    }],
    "temperature": 0.0,  # deterministic for a Judge
}

print("✅ Grok config ready")

In [ ]:
# CELL 3: Initialize Vector DB & Define the Search Tool
#
# In AutoGen, agents need plain Python functions as tools.
# We define the in-memory Chroma DB here, then wrap the search
# logic in a function the Retriever Agent can call.

from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Our fake company knowledge base — three overlapping facts.
# The trick question targets TUESDAY, but a naive retrieval
# might surface the MARCH entry first, causing the Judge to REJECT.
docs = [
    Document(page_content="On March 12th, the server outage was caused by a DDoS attack on the EU-East region."),
    Document(page_content="Tuesday's 504 Gateway Timeout was traced back to a misconfigured API limit in the EU-West load balancer."),
    Document(page_content="To fix an API limit timeout, engineers must increase the concurrent connection threshold in the AWS console."),
]

vectorstore = Chroma.from_documents(documents=docs, embedding=embeddings)


def search_database(query: str) -> str:
    """Searches the internal documentation for the given query and returns facts."""
    print(f"\n[TOOL EXECUTION] Searching DB for: '{query}'")
    results = vectorstore.similarity_search(query, k=2)
    context = "\n".join([doc.page_content for doc in results])
    return f"Retrieved Context:\n{context}"


print("✅ Vector DB initialised with 3 documents")
print("✅ search_database() tool defined")

In [ ]:
# CELL 4: Define the Agents
#
# Four distinct personas:
#   User_Proxy     — entry point (no LLM, just routes the initial message)
#   Retriever_Agent — armed with search_database(), finds facts
#   Generator_Agent — writer; produces a draft from chat history only
#   Judge_Agent     — strict QA; outputs APPROVE or REJECT
#
# The Judge's exact vocabulary ("REJECT" / "APPROVE") is what our
# FSM function in Cell 5 watches for.

from autogen import AssistantAgent, UserProxyAgent, register_function

# 1. The Proxy (human-side stand-in; fires once, then stays quiet)
user_proxy = UserProxyAgent(
    name="User_Proxy",
    system_message="A proxy for the user.",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=0
)

# 2. The Retriever — knows about the search tool
retriever_agent = AssistantAgent(
    name="Retriever_Agent",
    llm_config=grok_config,
    system_message=(
        "You are the Researcher. When asked a question, use the `search_database` tool "
        "to find facts. Output the raw facts you find, then say 'HANDOFF TO GENERATOR'."
    )
)

# Register search_database so the Retriever can call it
# (user_proxy executes the tool call on behalf of the group)
register_function(
    search_database,
    caller=retriever_agent,
    executor=user_proxy,
    name="search_database",
    description="Search the company database."
)

# 3. The Generator — writer, MUST ground to chat history only
generator_agent = AssistantAgent(
    name="Generator_Agent",
    llm_config=grok_config,
    system_message=(
        "You are the Writer. Read the chat history. "
        "Using ONLY the facts provided by the Retriever_Agent, "
        "write a clear, concise draft answering the user's original question."
    )
)

# 4. The Judge — gating logic lives in the system prompt
judge_agent = AssistantAgent(
    name="Judge_Agent",
    llm_config=grok_config,
    system_message=(
        "You are the strict QA Judge. Read the user's original question, "
        "the retrieved facts, and the Generator's draft.\n"
        "1. If the draft hallucinates or misses info, output 'REJECT:' followed by "
        "   instructions on what the Retriever needs to search for next.\n"
        "2. If the draft is perfect, output exactly 'APPROVE: TERMINATE'."
    )
)

print("✅ All four agents created")

In [ ]:
# CELL 5: The Orchestrator — State Routing Logic (the FSM)
#
# This is the heart of the Microsoft AutoGen approach.
# Instead of drawing graph edges (à la LangGraph), we write
# a plain Python function that reads the last message and
# returns whoever should speak next.
#
# The routing table:
#   User_Proxy       → Retriever_Agent  (always start with retrieval)
#   Retriever_Agent  → Generator_Agent  (hand off facts to the writer)
#   Generator_Agent  → Judge_Agent      (submit draft for evaluation)
#   Judge_Agent      → Retriever_Agent  (if REJECT — loop back!)
#   Judge_Agent      → None             (if APPROVE — end the chat)

from autogen import GroupChat, GroupChatManager


def state_transition(last_speaker, groupchat):
    messages = groupchat.messages
    if not messages:
        return retriever_agent

    last_message = messages[-1]["content"]

    if last_speaker == user_proxy:
        return retriever_agent

    elif last_speaker == retriever_agent:
        return generator_agent

    elif last_speaker == generator_agent:
        return judge_agent

    elif last_speaker == judge_agent:
        if "REJECT" in last_message:
            print("\n🚨 [GATE FAILED] Judge rejected the draft. Looping back to Retriever! 🚨")
            return retriever_agent  # ← the self-correcting loop
        elif "APPROVE" in last_message:
            print("\n✅ [GATE PASSED] Output is verified.")
            return None  # ends the group chat

    return None


group_chat = GroupChat(
    agents=[user_proxy, retriever_agent, generator_agent, judge_agent],
    messages=[],
    max_round=12,
    speaker_selection_method=state_transition
)

manager = GroupChatManager(groupchat=group_chat, llm_config=grok_config)

print("✅ GroupChat + FSM manager ready")

In [ ]:
# CELL 6: Run the Agentic RAG Loop
#
# We ask the trick question: "Tuesday" vs "March".
# Watch the terminal output — you are reading a live transcript
# of four agents talking to each other:
#
#   Retriever posts DB results into the chat.
#   Generator reads the chat and posts a draft.
#   Judge reads the draft and (possibly) fires back a critique.
#   Retriever loops again with improved search terms.
#
# In LangGraph you watched a dict update silently.
# Here you watch employees argue until they get it right.

query = "What caused Tuesday's 504 timeout and in which region did it happen?"

print("=" * 60)
print("STARTING MICROSOFT AUTOGEN SWARM")
print("=" * 60)

chat_result = user_proxy.initiate_chat(
    manager,
    message=query
)

print("\n" + "=" * 60)
print("FINAL CHAT HISTORY ENDED SUCCESSFULLY")
print("=" * 60)

---

## 🏗️ The Architectural Difference: Graph vs. Conversation

| | LangGraph | Microsoft AutoGen |
|---|---|---|
| **Mental model** | Data pipeline | Employees in a group chat |
| **Routing** | Edges + conditional branches | Python function reading the last message |
| **Observability** | Dict values change silently | You read a full transcript |
| **Self-correction** | Cycle in the graph | FSM returns original speaker |
| **Best for** | Deterministic, data-heavy pipelines | Emergent, multi-persona reasoning |

Microsoft's defining philosophy: treat LLMs as **distinct, interacting personas** rather than function nodes. The conversation IS the state machine.

### Next steps
- Swap Grok for any OpenAI-compatible model by updating `grok_config`
- Add more agents (e.g., a `Fact_Checker` between Generator and Judge)
- Replace `Chroma` with a real database by swapping `search_database()`
- Explore [Magentic-One](https://microsoft.github.io/autogen/stable/user-guide/agentchat-user-guide/magentic-one.html) for more complex multi-agent orchestration
